# GeoZones mapping — bbox variant

Same pipeline as `geozones-mapping.ipynb` but using precomputed zone bounding boxes instead
of actual geometries. Dataset geometries are also reduced to their bounding box.

**Premise**: when a dataset's spatial bbox was set to cover a specific zone, it should be
nearly identical to that zone's bounding box. We therefore compare two rectangles with a high
IoU threshold (0.8). The IoU is computed with pure arithmetic — no `shapely.intersection` needed.

In [1]:
import geopandas as gpd
import pandas as pd
import shapely
import numpy as np
import json, os
from shapely.geometry import shape, box

IOU_THRESHOLD = 0.8

ZONES_BBOXES = "./zones_bboxes.json"
DATASET_CSV  = "./export-dataset.csv"
OUT_CSV      = "./geozones-mapping-bboxes-output.csv"
CRS_GEO      = "EPSG:4326"

In [2]:
## Load zone bboxes → GeoDataFrame of box geometries

with open(ZONES_BBOXES) as f:
    raw = json.load(f)

zones = gpd.GeoDataFrame(
    {"zone_id": list(raw.keys()),
     "geometry": [box(*b) for b in raw.values()]},
    crs=CRS_GEO,
)
zones["zone_area"] = shapely.area(zones.geometry.values)
print(f"Zones: {len(zones):,}")

Zones: 38,340


In [3]:
## Load datasets → reduce geometry to bbox

df = pd.read_csv(DATASET_CSV, sep=";", dtype=str)

def parse_geom(raw):
    if not isinstance(raw, str) or not raw.strip():
        return None
    try:
        return shape(json.loads(raw))
    except Exception:
        return None

df["geometry"] = df["spatial.geom"].apply(parse_geom)
datasets = gpd.GeoDataFrame(df, geometry="geometry", crs=CRS_GEO)
has_geom = datasets[datasets.geometry.notna()].copy()

# Ensure all dataset geometries are bboxes
has_geom["geometry"] = shapely.envelope(has_geom.geometry.values)
has_geom["bbox_area"] = shapely.area(has_geom.geometry.values)

print(f"Datasets with geometry: {len(has_geom):,}")

Datasets with geometry: 45,431


In [4]:
import time

sample_geo = has_geom[["id", "geometry", "bbox_area"]].copy()

t0 = time.perf_counter()

candidates = gpd.sjoin(
    sample_geo,
    zones[["zone_id", "zone_area", "geometry"]],
    how="inner",
    predicate="intersects",
).drop(columns="index_right")

t1 = time.perf_counter()
print(f"sjoin: {t1-t0:.1f}s  — {len(candidates):,} candidate pairs")

# Area pre-filter (lossless)
lo = np.minimum(candidates["zone_area"].values, candidates["bbox_area"].values)
hi = np.maximum(candidates["zone_area"].values, candidates["bbox_area"].values)
plausible = candidates[(hi > 0) & (lo / hi >= IOU_THRESHOLD)].copy()
print(f"Area pre-filter: {len(candidates):,} → {len(plausible):,} candidates")

# Bbox IoU — pure arithmetic, no shapely.intersection needed
zone_geom_lookup = zones.set_index("zone_id")["geometry"]
d_bounds = shapely.bounds(plausible.geometry.values)
z_bounds = shapely.bounds(zone_geom_lookup.reindex(plausible["zone_id"]).values)

inter_minx = np.maximum(d_bounds[:, 0], z_bounds[:, 0])
inter_miny = np.maximum(d_bounds[:, 1], z_bounds[:, 1])
inter_maxx = np.minimum(d_bounds[:, 2], z_bounds[:, 2])
inter_maxy = np.minimum(d_bounds[:, 3], z_bounds[:, 3])
inter_area = np.maximum(0, inter_maxx - inter_minx) * np.maximum(0, inter_maxy - inter_miny)
union_area  = plausible["bbox_area"].values + plausible["zone_area"].values - inter_area
plausible["iou_score"] = np.where(union_area > 0, inter_area / union_area, 0.0)

t2 = time.perf_counter()
print(f"Bbox IoU: {t2-t1:.1f}s")

best = (
    plausible[plausible["iou_score"] >= IOU_THRESHOLD]
    .sort_values("iou_score", ascending=False)
    .drop_duplicates(subset="id", keep="first")
    [["id", "zone_id", "iou_score"]]
)

t3 = time.perf_counter()
print(f"\nTotal: {t3-t0:.0f}s — {len(best):,} matched / {len(has_geom):,} datasets with geometry")

sjoin: 24.3s  — 133,009,435 candidate pairs


Area pre-filter: 133,009,435 → 101,438 candidates
Bbox IoU: 3.1s

Total: 27s — 29,219 matched / 45,431 datasets with geometry


In [5]:
output = best.merge(has_geom[["id", "title"]], on="id", how="left")[["id", "title", "zone_id", "iou_score"]]
output.to_csv(OUT_CSV, index=False)
print(f"Saved {len(output):,} rows → {OUT_CSV}")

output["level"] = output["zone_id"].str.extract(r"^([^:]+:[^:]+)")
print(output["level"].value_counts().to_string())

Saved 29,219 rows → ./geozones-mapping-bboxes-output.csv


level
fr:departement         14672
fr:commune              7495
fr:region               5298
fr:epci                  906
country-subset:fr        767
country-group:world       74
country:fr                 7
